# Compressing Initialized Memory


In [1]:
from notebook_imports import *

/Users/matthewho/miniconda3/envs/arc_agi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from concept_mem.memory.v4.concept import Concept
from concept_mem.memory.v4.memory import ConceptMemory
from concept_mem.utils import extract_yaml_block

In [3]:
memory_abstraction_output_dir = REPO_ROOT / "outputs/2025-07-24/23-32-57"
cm = ConceptMemory()
cm.load_from_file(memory_abstraction_output_dir / "memory.json")

In [4]:
requires_compression = []
for c, info in cm.concepts.items():
    if len(info.used_in) <= 1:
        continue
    if len(info.cues) > 1 or len(info.implementation) > 1:
        requires_compression.append(info)
print(len(requires_compression))

93


In [5]:
llm_client = LLMClient(
    provider=Provider.OPENAI,
    cache_dir=str(REPO_ROOT / "cache"),
    dotenv_path=DOTENV_PATH,
)

gen_cfg = GenerationConfig(
    n=1,
    temperature=0.1,
    max_tokens=1024,
    top_p=1,
    batch_size=16,
    seed=88,
    ignore_cache=False,
    expand_multi=None,  # let the provider decide
)

In [6]:
compression_prompt_template = """\
# Introduction
Consider a class of "ARC" puzzles where each puzzle has a hidden transformation rule that maps input grids to output grids. Each puzzle presents several input-output grid pairs as reference examples and the task is to predict the transformation rule. Grids are 2D numpy integer arrays with integers representing colors. 0 represents black and usually serves as the background.

We are trying to learn from previously solved puzzles to help solve more puzzles in the future. We analyzed puzzle solutions and abstracted reusable concepts. Here, a concept can encode one of the following:
1. routine: routines from the solution program, which can either directly output a grid, or prepare handle some intermediate processing
2. structure: a class of visual entities (not program data structures) that can be seen in an individual pixel grid. The visual entity should be general, perhaps by being defined by their function or in relation to an operation.

Following a functional programming philosophy, we promote reusability through composition by parameterizing routines:
- We also encourage passing specialized logic (other routines) as parameters to higher order routines
- Each routine is typed-- it specifies output typing in addition to a list of parameter specifications
- We allow custom types to be defined with the format "name := definition"
- Often custom types are `Callable` that specify some pluggable operations

The overarching goal is to help future puzzle solving which boils down to 2 problems:
1. determining what the transformation is (by examining input/output examples)
2. implementing the transformation in code

- These concepts are meant to compactly encode ideas from solved puzzles and help puzzle solving by remembering what operations could be involved and how they might interact with each other via typing. 
- Parameterization and composition via higher order routines/pluggable operations helps ensure concepts are reusable and not overly specific to one puzzle
- But to specifically address the 2 main problems, we also annotate concepts with:
    1. relevance cues: suggestion of what to look for at puzzle solving time that would indicate this concept is potentially relevant.
        - can either describe (1) things to look for in the puzzle grids or (2) related concepts that might implicate this one
        - we primarily care about annotating cues for grid manipulation routines and structures where there are concrete things to look for in the reference grids.
    2. implementation notes: suggestions on how to implement the idea programmatically.

# Task
We allowed multiple passes to add to the cues and implementation notes lists, and now we are looking to remove redundancy. Make sure to keep separate ideas in separate entries, but remove duplicate entries in a list or if they are very similar and only subtly different, try to synthesize into a single entry.

We expect you to output a fenced yaml markdown block (preceded by "```yaml" and followed by "```") that contains a re-written version of the cues and implementation notes lists. Here's how we expect the output to look like:
```yaml
cues:
  - first cue
  - second cue
implementation:
  - first implementation note
  - second implementation note
```

# Your Concept Annotation
```
{concept}
```"""

In [ ]:
# generate compressed annotations
prompts = [
    compression_prompt_template.format(concept=c.to_string())
    for c in requires_compression
]
res = await llm_client.async_batch_generate(
    prompts=prompts,
    model="gpt-4.1-2025-04-14",
    gen_cfg=gen_cfg,
)
write_json(
    res,
    REPO_ROOT / "data/abstract_anno/op3/cue_impl_compressed_v2.json",
)

In [ ]:
# write compressed cues and implementation notes back to concepts
for c, model_out in zip(requires_compression, res):
    yaml_string = extract_yaml_block(model_out[0])
    cue_impl = yaml.safe_load(yaml_string)
    assert isinstance(cue_impl, dict), f"Expected dict, got {type(cue_impl)}"
    c.cues = cue_impl.get("cues", c.cues)
    c.implementation = cue_impl.get("implementation", c.implementation)

In [ ]:
# save compressed memory
cm.save_to_file(DATA_DIR / "memory/compressed_v1.json")